# ROGII Mitch 9.398 clean inference

Inference-only private fork of the public Mitch Gansemer notebook. Skips demo/writeup cells and writes a hidden-safe `submission.csv`.


In [ ]:
import sys, os, json, time
import numpy as np
import pandas as pd
import joblib
import matplotlib.pyplot as plt
from pathlib import Path
from scipy.signal import savgol_filter
from scipy.optimize import minimize
from sklearn.model_selection import GroupKFold
from tqdm import tqdm

import lightgbm as lgb
import xgboost as xgb
from catboost import CatBoostRegressor

# ── On-Kaggle vs local detection ──────────────────────────────────────────────
_ON_KAGGLE = os.path.exists('/kaggle')

# ── Locate MODEL_DIR (contains feature_cols.json, fold models, KNN artifacts) ─
MODEL_DIR = Path('models')
if _ON_KAGGLE:
    for _root, _dirs, _files in os.walk('/kaggle/input'):
        if 'feature_cols.json' in _files:
            MODEL_DIR = Path(_root)
            break
print(f'ON_KAGGLE={_ON_KAGGLE}  MODEL_DIR={MODEL_DIR}')

# ── Add utils.py to path ───────────────────────────────────────────────────────
sys.path.insert(0, str(MODEL_DIR) if _ON_KAGGLE else '.')

import utils
from utils import (
    load_typewell, load_horizontal, get_train_well_ids, get_test_well_ids,
    rmse, set_style, save_fig,
    impute_gr_with_typewell, build_typewell_interp,
    gr_xcorr_batch, multi_scale_ncc, multi_scale_ncc_anchor,
    FormationKNN, FormationPlaneKNN, RowKNN, DenseANCCImputer,
    visible_gr_shift_fit, viterbi_tvt, particle_filter_tvt, particle_filter_ancc,
    learn_z_beta, BEAM_VARIANTS, PF_VARIANTS,
    compute_trajectory_kinematics, estimate_apparent_dip,
    FORMATION_COLS,
)
set_style()

# On Kaggle: update data dirs to competition input path
if _ON_KAGGLE:
    _data_candidates = [
        Path('/kaggle/input/competitions/rogii-wellbore-geology-prediction'),
        Path('/kaggle/input/rogii-wellbore-geology-prediction'),
    ]
    for _p in _data_candidates:
        if (_p / 'train').exists() and (_p / 'test').exists() and (_p / 'sample_submission.csv').exists():
            utils.DATA_DIR = _p
            break
    else:
        for _sample in Path('/kaggle/input').glob('**/sample_submission.csv'):
            _p = _sample.parent
            if (_p / 'train').exists() and (_p / 'test').exists():
                utils.DATA_DIR = _p
                break
        else:
            raise FileNotFoundError('Could not locate competition train/test/sample_submission under /kaggle/input')
    utils.TRAIN_DIR = utils.DATA_DIR / 'train'
    utils.TEST_DIR = utils.DATA_DIR / 'test'
print(f'DATA_DIR={utils.DATA_DIR}')

# ── Inference constants (must match 02_features.ipynb exactly) ────────────────
XCORR_WINDOW   = 100
XCORR_RADIUS   = 50.0
XCORR_STEP     = 1.0
XCORR_STRIDE   = 10
ANCHOR_TAIL    = 200
NCC_RADIUS     = 150.0
NCC_HALFWIDTHS = (8, 15, 25)
NCC_TEMP       = 10.0
ROLL_SHORT     = 25
ROLL_LONG      = 100
KNN_K          = 15
BEAM_RADIUS    = 80.0
PF_PARTICLES   = 500
TW_OFFSETS     = np.array([-80, -40, -20, -10, -5, 0, 5, 10, 20, 40, 80], dtype=np.float32)
SG_W    = 17   # Savitzky-Golay window (R8: domain prior from reference notebook)
SG_P    = 3
N_FOLDS = 5
SEED    = 42

# ── Load serialized KNN structures (needed for demos and inference) ────────────
t0 = time.time()
formation_plane_knn = joblib.load(MODEL_DIR / 'formation_plane_knn.joblib')
formation_knn       = joblib.load(MODEL_DIR / 'formation_knn.joblib')
row_knn             = joblib.load(MODEL_DIR / 'row_knn.joblib')
dense_imputer       = joblib.load(MODEL_DIR / 'dense_imputer.joblib')
print(f'KNN artifacts loaded in {time.time()-t0:.1f}s')



In [ ]:
with open(MODEL_DIR / 'blend_weights.json') as f:
    BLEND_WEIGHTS = json.load(f)
print('Blend weights:', {k: f'{v:.4f}' for k, v in BLEND_WEIGHTS.items()})


In [13]:
TW_OFFSETS = np.array([-80, -40, -20, -10, -5, 0, 5, 10, 20, 40, 80], dtype=np.float32)

def build_features(well_id: str, split: str = 'train', loo: bool = True) -> pd.DataFrame:
    """
    Build eval-zone features for one well.
    loo=True excludes this well from the formation KNN (use for training wells).
    """
    tw = load_typewell(well_id, split=split)
    hw = load_horizontal(well_id, split=split)

    anchor_mask = hw['TVT_input'].notna()
    anchor = hw[anchor_mask].copy().reset_index(drop=True)
    eval_z = hw[~anchor_mask].copy().reset_index(drop=True)

    if len(eval_z) == 0:
        return pd.DataFrame()

    # ---- Anchor zone summary ----
    last_tvt  = float(anchor['TVT_input'].iloc[-1])
    last_md   = float(anchor['MD'].iloc[-1])
    last_z    = float(anchor['Z'].iloc[-1])
    n_anchor  = len(anchor)

    tail = anchor.tail(ANCHOR_TAIL)
    if len(tail) >= 2:
        dtvt     = tail['TVT_input'].iloc[-1] - tail['TVT_input'].iloc[0]
        dmd_tail = tail['MD'].iloc[-1] - tail['MD'].iloc[0]
        tvt_rate = dtvt / dmd_tail if abs(dmd_tail) > 1e-6 else 0.0
    else:
        tvt_rate = 0.0

    md_from_anchor = eval_z['MD'].values - last_md
    tvt_extrap     = last_tvt + tvt_rate * md_from_anchor

    if n_anchor >= 2:
        tvt_step_per_row = abs(
            (anchor['TVT_input'].iloc[-1] - anchor['TVT_input'].iloc[0]) / (n_anchor - 1)
        )
        tvt_step_per_row = max(0.05, tvt_step_per_row)
    else:
        tvt_step_per_row = 1.0

    _tail_for_rate = anchor.tail(ANCHOR_TAIL)
    if len(_tail_for_rate) >= 2:
        _tail_tvt_diff = abs(
            _tail_for_rate['TVT_input'].iloc[-1] - _tail_for_rate['TVT_input'].iloc[0]
        )
        tail_tvt_step = max(0.001, _tail_tvt_diff / (len(_tail_for_rate) - 1))
    else:
        tail_tvt_step = 0.01

    # ---- Prefix features (anchor tail TVT behavior) ----
    anchor_tvt_vals = anchor['TVT_input'].values
    anchor_md_vals  = anchor['MD'].values
    anchor_z_vals   = anchor['Z'].values

    def _tail_mean_diff(vals, n):
        v = vals[-(n + 1):]
        return float(np.diff(v).mean()) if len(v) >= 2 else 0.0

    def _tail_slope(y, x, n):
        y, x = y[-n:], x[-n:]
        if len(y) < 2: return 0.0
        cx = x - x.mean()
        d  = float(np.dot(cx, cx))
        return float(np.dot(cx, y - y.mean()) / d) if d > 0 else 0.0

    prefix_tvt_step20       = _tail_mean_diff(anchor_tvt_vals, 20)
    prefix_tvt_step100      = _tail_mean_diff(anchor_tvt_vals, 100)
    prefix_tvt_md_slope100  = _tail_slope(anchor_tvt_vals, anchor_md_vals, 100)

    # ---- GR imputation ----
    tw_sorted = tw.sort_values('TVT').dropna(subset=['GR'])
    tw_tvt    = tw_sorted['TVT'].values
    tw_gr     = tw_sorted['GR'].values
    tw_interp = build_typewell_interp(tw)

    gr_full   = impute_gr_with_typewell(hw, tw).values.astype(float)
    gr_eval   = eval_z['GR'].copy().values.astype(float)
    null_mask = np.isnan(gr_eval)
    if null_mask.any():
        gr_eval[null_mask] = tw_interp(tvt_extrap[null_mask])
    gr_full[n_anchor:] = gr_eval

    anchor_gr_vals = gr_full[:n_anchor]
    finite_gr      = anchor_gr_vals[np.isfinite(anchor_gr_vals)]
    p5, p95        = (np.nanpercentile(finite_gr, [5, 95]) if len(finite_gr) >= 10
                      else (finite_gr.min(), finite_gr.max()))
    gr_norm_p5p95  = (gr_eval - p5) / max(p95 - p5, 1.0)

    # ---- Fractional position in eval zone ----
    n_eval  = len(eval_z)
    frac    = np.arange(n_eval) / max(n_eval - 1, 1)
    frac2   = frac ** 2
    sqrt_frac = np.sqrt(frac)

    # ---- GR signal features (1st/2nd derivative, envelope) ----
    gr_d1  = np.diff(gr_eval, prepend=gr_eval[0]).astype(np.float32)
    gr_d2  = np.diff(gr_d1,   prepend=gr_d1[0]).astype(np.float32)
    gr_env = (pd.Series(gr_eval)
               .rolling(25, center=True, min_periods=1).max()
               .values.astype(np.float32))

    # ---- Formation KNN: plane-fit (new primary) + legacy IDW ----
    x_ev = eval_z['X'].values
    y_ev = eval_z['Y'].values
    z_ev = eval_z['Z'].values

    exclude = well_id if loo else None
    xy_anchor_q = np.column_stack([anchor['X'].values, anchor['Y'].values])
    xy_eval_q   = np.column_stack([x_ev, y_ev])

    # Plane-fit KNN (new primary spatial imputer)
    form_eval_plane,   knn_dist_plane = formation_plane_knn.predict(xy_eval_q,   exclude_wid=exclude)
    form_anchor_plane, _              = formation_plane_knn.predict(xy_anchor_q, exclude_wid=exclude)

    # Legacy IDW KNN (kept for backward compat)
    form_eval_idw,   knn_dist_idw = formation_knn.predict(xy_eval_q,   exclude_wid=exclude)
    form_anchor_idw, _            = formation_knn.predict(xy_anchor_q, exclude_wid=exclude)

    # Use plane-fit as primary source for b_well offsets
    form_anchor_pred = form_anchor_plane
    form_eval_pred   = form_eval_plane
    knn_dist         = knn_dist_plane

    # ---- Per-formation segmented b_well offsets ----
    form_b = {}
    for fi, fn in enumerate(FORMATION_COLS):
        bv   = anchor_tvt_vals + anchor_z_vals - form_anchor_pred[:, fi]
        n_bv = len(bv)
        t1, t2 = n_bv // 3, 2 * n_bv // 3
        b_full  = float(np.median(bv))
        b_late  = float(np.median(bv[-50:] if n_bv >= 50 else bv))
        b_early = float(np.median(bv[:max(1, t1)]) if t1 > 0 else b_full)
        b_mid   = float(np.median(bv[t1:max(t1 + 1, t2)]) if t2 > t1 else b_full)
        w_exp   = np.exp(0.02 * np.arange(n_bv))
        b_wls   = float(np.dot(w_exp / w_exp.sum(), bv))
        form_b[fn] = dict(full=b_full, early=b_early, mid=b_mid, late=b_late, wls=b_wls)

    tvt_form_feats = {}
    for fi, fn in enumerate(FORMATION_COLS):
        fb = form_b[fn]
        tvt_form_feats[f'tvtF_{fn}']   = (-z_ev + form_eval_pred[:, fi] + fb['full']).astype(np.float32)
        tvt_form_feats[f'tvtFw_{fn}']  = (-z_ev + form_eval_pred[:, fi] + fb['wls']).astype(np.float32)
        tvt_form_feats[f'tvtF50_{fn}'] = (-z_ev + form_eval_pred[:, fi] + fb['late']).astype(np.float32)

    b_full  = form_b['ANCC']['full']
    b_late  = form_b['ANCC']['late']
    b_wls   = form_b['ANCC']['wls']
    b_early = form_b['ANCC']['early']
    b_mid   = form_b['ANCC']['mid']
    ancc_eval  = form_eval_pred[:, 0]
    egfdu_eval = form_eval_pred[:, 3]
    buda_eval  = form_eval_pred[:, 5]
    tvt_form_full  = tvt_form_feats['tvtF_ANCC']
    tvt_form_late  = tvt_form_feats['tvtF50_ANCC']
    tvt_form_wls   = tvt_form_feats['tvtFw_ANCC']
    tvt_form_egfdu = tvt_form_feats['tvtF_EGFDU']
    tvt_form_buda  = tvt_form_feats['tvtF_BUDA']

    # ---- RowKNN: row-level ANCC imputer ----
    row_ancc_eval, row_ancc_std_eval, row_dist_eval = row_knn.predict(xy_eval_q,   exclude_wid=exclude)
    row_ancc_anch, _,                 _             = row_knn.predict(xy_anchor_q, exclude_wid=exclude)

    # b_well offset from RowKNN
    if n_anchor > 0:
        b_per_row_row = anchor_tvt_vals + anchor_z_vals - row_ancc_anch.astype(np.float64)
        w_row = np.exp(0.02 * np.arange(len(b_per_row_row)))
        b_well_row = float(np.dot(w_row / w_row.sum(), b_per_row_row))
    else:
        b_well_row = 0.0
    knn_row_tvt_pred = -z_ev + row_ancc_eval.astype(np.float64) + b_well_row
    knn_row_tvt_pred_delta = (knn_row_tvt_pred - last_tvt).astype(np.float32)

    # ---- DenseANCCImputer ----
    dense_ancc_eval, dense_dist_eval, dense_std_eval = dense_imputer.predict(xy_eval_q, exclude_wid=exclude)
    dense_ancc_anch, _, _                            = dense_imputer.predict(xy_anchor_q, exclude_wid=exclude)

    if n_anchor > 0:
        b_per_row_dense = anchor_tvt_vals + anchor_z_vals - dense_ancc_anch.astype(np.float64)
        w_den = np.exp(0.02 * np.arange(len(b_per_row_dense)))
        b_well_dense = float(np.dot(w_den / w_den.sum(), b_per_row_dense))
    else:
        b_well_dense = 0.0
    dense_tvt_pred = -z_ev + dense_ancc_eval.astype(np.float64) + b_well_dense
    dense_tvt_pred_delta = (dense_tvt_pred - last_tvt).astype(np.float32)

    # ---- Rolling GR ----
    gr_series   = pd.Series(gr_full)
    roll_s_mean = gr_series.rolling(ROLL_SHORT, center=True, min_periods=1).mean().values
    roll_s_std  = gr_series.rolling(ROLL_SHORT, center=True, min_periods=1).std().fillna(0).values
    roll_l_mean = gr_series.rolling(ROLL_LONG,  center=True, min_periods=1).mean().values
    roll_l_std  = gr_series.rolling(ROLL_LONG,  center=True, min_periods=1).std().fillna(0).values

    gr_tw_last_anchor = float(tw_interp(last_tvt))
    gr_diff           = gr_eval - gr_tw_last_anchor

    # ---- xcorr ----
    xcorr_tvt, xcorr_conf = gr_xcorr_batch(
        gr_full=gr_full, n_anchor=n_anchor, tvt_seeds=tvt_extrap,
        tw_tvt=tw_tvt, tw_gr=tw_gr, tvt_step_per_row=tail_tvt_step,
        window_rows=XCORR_WINDOW, search_radius=XCORR_RADIUS,
        xcorr_step=XCORR_STEP, stride=XCORR_STRIDE,
    )
    xcorr_delta    = xcorr_tvt - tvt_extrap
    xcorr_smooth   = pd.Series(xcorr_tvt.astype(float)).rolling(10, center=True, min_periods=1).mean().values
    tw_gr_at_xcorr = tw_interp(xcorr_tvt).astype(float)
    xcorr_residual = gr_eval - tw_gr_at_xcorr

    # ---- Visible GR shift ----
    gr_shift_ft, gr_shift_corr, gr_shift_bias = visible_gr_shift_fit(
        anchor_tvt_vals, anchor_gr_vals, tw_tvt, tw_gr,
    )

    # ---- Multi-scale NCC vs typewell (original, per-row seeds) ----
    ncc_tvt, ncc_conf = multi_scale_ncc(
        eval_gr=gr_eval, anchor_gr=anchor_gr_vals,
        tw_tvt=tw_tvt, tw_gr=tw_gr,
        last_anchor_tvt=last_tvt, tvt_rate=tail_tvt_step,
        search_radius=NCC_RADIUS, half_widths=NCC_HALFWIDTHS, temperature=NCC_TEMP,
    )
    ncc_delta = (ncc_tvt - last_tvt).astype(np.float32)

    # ---- Multi-scale NCC vs ANCHOR zone (new: anchor GR as template) ----
    ncc_anchor_tvt, ncc_anchor_conf = multi_scale_ncc_anchor(
        anchor_gr=anchor_gr_vals.astype(np.float32),
        anchor_tvt=anchor_tvt_vals.astype(np.float32),
        eval_gr=gr_eval.astype(np.float32),
        half_widths=NCC_HALFWIDTHS,
        stride=3,
    )
    ncc_anchor_delta = (ncc_anchor_tvt - last_tvt).astype(np.float32)

    # ---- Softmax blend: NCC × formation ----
    _ncc_w = 1.0 / (1.0 + np.exp(-3.0 * ncc_conf.astype(np.float64)))
    ncc_form_blend = (_ncc_w * ncc_tvt + (1.0 - _ncc_w) * tvt_form_feats['tvtFw_ASTNU']).astype(np.float32)
    ncc_form_delta = (ncc_form_blend - last_tvt).astype(np.float32)

    # ---- GR signal divergence: std across estimators ----
    pf_placeholder = np.full(n_eval, last_tvt, dtype=np.float32)  # will be filled after PF
    form_preds_stack = np.stack([tvt_form_feats[f'tvtF_{fn}'] for fn in FORMATION_COLS], axis=1)
    form_std = form_preds_stack.std(axis=1).astype(np.float32)

    # ---- Z-velocity physics prior ----
    z_beta, z_intercept, _ = learn_z_beta(anchor_tvt_vals, anchor_z_vals, anchor_md_vals)

    # ---- GR for beam/PF ----
    _gr_raw = eval_z['GR'].copy().values.astype(float)
    gr_beam = (pd.Series(_gr_raw)
               .interpolate(limit_direction='both')
               .fillna(float(np.nanmean(tw_gr)))
               .rolling(5, center=True, min_periods=1).mean()
               .values.astype(np.float32))

    # ---- Viterbi beam search (4 variants) ----
    beam_results = {}
    for name, emit_s, move_s in BEAM_VARIANTS:
        path = viterbi_tvt(
            gr_beam, tw_tvt, tw_gr,
            last_tvt, tvt_step_per_row,
            emit_sigma=emit_s, move_sigma=move_s,
            search_radius=BEAM_RADIUS, grid_step=1.0,
        )
        beam_results[f'{name}_delta'] = (path - last_tvt).astype(np.float32)

    # ---- Particle filter: TVT-based (2 variants) ----
    eval_z_vals = eval_z['Z'].values.astype(np.float64)
    eval_md_vals = eval_z['MD'].values.astype(np.float64)

    pf_results = {}
    for name, mom, gr_s, jit_s in PF_VARIANTS:
        est = particle_filter_tvt(
            gr_beam, tw_tvt, tw_gr,
            last_tvt, tail_tvt_step,
            n_particles=PF_PARTICLES, momentum=mom,
            gr_sigma=gr_s, jitter_sigma=jit_s, seed=42,
        )
        pf_results[f'{name}_delta'] = (est - last_tvt).astype(np.float32)

    # ---- Particle filter: ANCC-based ----
    pf_ancc_tvt, pf_ancc_std = particle_filter_ancc(
        eval_gr=gr_beam.astype(np.float64),
        eval_z=eval_z_vals,
        eval_md=eval_md_vals,
        tw_tvt=tw_tvt, tw_gr=tw_gr,
        last_anchor_tvt=last_tvt,
        last_anchor_z=last_z,
        last_anchor_md=last_md,
        anchor_tvt=anchor_tvt_vals,
        anchor_z=anchor_z_vals,
        anchor_md=anchor_md_vals,
        n_particles=PF_PARTICLES,
        seed=42,
    )
    pf_ancc_delta = (pf_ancc_tvt - last_tvt).astype(np.float32)

    # ---- Signal divergence (after PF is available) ----
    pf_z_est  = last_tvt + pf_results['pf_z_delta']
    sig_std   = np.stack([
        pf_z_est,
        last_tvt + beam_results['beam_med2_delta'],
        ncc_anchor_tvt,
        tvt_form_feats['tvtF_ANCC'],
    ], axis=1).std(axis=1).astype(np.float32)

    pf_vs_spatial = (pf_ancc_delta - (form_std / (form_std.mean() + 1e-3))).astype(np.float32)
    sc_vs_beam    = (ncc_anchor_delta - beam_results['beam_med2_delta']).astype(np.float32)

    # ---- Typewell GR residuals anchored at NCC-anchor estimate ----
    tdsc_offsets = [-30, -15, -8, -4, -2, 0, 2, 4, 8, 15, 30]
    tdsc_feats = {
        f'tdsc_{int(o)}': (gr_eval - np.interp(
            ncc_anchor_tvt + o, tw_tvt, tw_gr
        )).astype(np.float32)
        for o in tdsc_offsets
    }

    # ---- Typewell GR offset dictionary ----
    tw_diff = {
        f'tw_diff_{int(off)}': (gr_eval - float(tw_interp(last_tvt + float(off)))).astype(np.float32)
        for off in TW_OFFSETS
    }

    # ---- Geometry ----
    md_ev  = eval_z['MD'].values
    hw_md  = hw['MD'].values
    hw_x, hw_y, hw_z = hw['X'].values, hw['Y'].values, hw['Z'].values
    dmd    = np.where(np.abs(np.diff(hw_md, prepend=hw_md[0])) < 1e-6, np.nan,
                      np.diff(hw_md, prepend=hw_md[0]))
    dz_dmd = pd.Series(np.diff(hw_z, prepend=hw_z[0]) / dmd).bfill().ffill().values
    dx_dmd = pd.Series(np.diff(hw_x, prepend=hw_x[0]) / dmd).bfill().ffill().values
    dy_dmd = pd.Series(np.diff(hw_y, prepend=hw_y[0]) / dmd).bfill().ffill().values

    # ---- Trajectory kinematics (inclination, azimuth, DLS, build rate) ----
    kin       = compute_trajectory_kinematics(hw_md, hw_x, hw_y, hw_z)
    incl_all  = kin['incl_deg']
    azi_all   = kin['azi_deg']
    dls_all   = kin['dls']
    build_all = kin['build_rate']

    incl_ev     = incl_all[n_anchor:]
    azi_ev      = azi_all[n_anchor:]
    dls_ev      = dls_all[n_anchor:]
    build_ev    = build_all[n_anchor:]
    cos_incl_ev = np.cos(np.radians(incl_ev)).astype(np.float32)
    sin_incl_ev = np.sin(np.radians(incl_ev)).astype(np.float32)

    # ---- Apparent dip from anchor zone kinematics ----
    dip = estimate_apparent_dip(anchor_tvt_vals, anchor_md_vals, incl_all[:n_anchor])
    b_dip_full  = float(dip['b_full'])
    b_dip_late  = float(dip['b_late'])
    b_dip_early = float(dip['b_early'])
    b_dip_slope = float(dip['b_slope'])

    # Physics TVT extrapolation: dTVT_i = (cos(incl_i) + sin(incl_i)*b) * dMD_i
    _dmd_steps = np.diff(eval_z['MD'].values, prepend=last_md)
    tvt_dip_full_arr = (last_tvt + np.cumsum(
        (cos_incl_ev + sin_incl_ev * np.float32(b_dip_full)) * _dmd_steps
    )).astype(np.float32)
    tvt_dip_late_arr = (last_tvt + np.cumsum(
        (cos_incl_ev + sin_incl_ev * np.float32(b_dip_late)) * _dmd_steps
    )).astype(np.float32)

    # Signed azimuth deviation from anchor circular mean
    _azi_anch_rad = np.radians(azi_all[:n_anchor])
    _azi_mean_rad = float(np.arctan2(np.sin(_azi_anch_rad).mean(), np.cos(_azi_anch_rad).mean()))
    azi_delta_ev  = np.degrees(np.arctan2(
        np.sin(np.radians(azi_ev) - _azi_mean_rad),
        np.cos(np.radians(azi_ev) - _azi_mean_rad),
    )).astype(np.float32)

    # Formation plane dip direction at well centroid (numerical gradient of plane-fit KNN)
    _xc, _yc = float(np.median(x_ev)), float(np.median(y_ev))
    _h = 500.0
    _fc  = formation_plane_knn.predict(np.array([[_xc,      _yc     ]]), exclude_wid=exclude)[0][0]
    _fdx = formation_plane_knn.predict(np.array([[_xc + _h, _yc     ]]), exclude_wid=exclude)[0][0]
    _fdy = formation_plane_knn.predict(np.array([[_xc,      _yc + _h]]), exclude_wid=exclude)[0][0]
    _ai  = FORMATION_COLS.index('ANCC')
    plane_dip_x = float((_fdx[_ai] - _fc[_ai]) / _h)  # d(ANCC_depth)/dX
    plane_dip_y = float((_fdy[_ai] - _fc[_ai]) / _h)  # d(ANCC_depth)/dY

    # Directional apparent dip: formation-deepening rate in wellbore azimuth direction
    apparent_dip_dir = (
        plane_dip_x * np.sin(np.radians(azi_ev)) +
        plane_dip_y * np.cos(np.radians(azi_ev))
    ).astype(np.float32)


    # ---- Well context ----
    anchor_gr_mean   = float(np.nanmean(anchor_gr_vals))
    anchor_gr_std    = float(np.nanstd(anchor_gr_vals)) + 1e-6
    anchor_tvt_mean  = float(np.nanmean(anchor_tvt_vals))
    anchor_tvt_range = float(np.nanmax(anchor_tvt_vals) - np.nanmin(anchor_tvt_vals))
    n_anchor_rows    = int(n_anchor)
    tw_in_range = tw_sorted[(tw_sorted['TVT'] >= anchor_tvt_vals.min()) &
                            (tw_sorted['TVT'] <= anchor_tvt_vals.max())]
    tw_gr_anchor_mean = float(tw_in_range['GR'].mean()) if len(tw_in_range) > 0 else anchor_gr_mean

    # ---- GR lag/lead (normalized per-well) ----
    gr_norm   = (gr_eval - anchor_gr_mean) / anchor_gr_std
    gr_norm_s = pd.Series(gr_norm)
    gr_lag_5   = gr_norm_s.shift(5).values;  gr_lag_10  = gr_norm_s.shift(10).values
    gr_lag_25  = gr_norm_s.shift(25).values; gr_lag_50  = gr_norm_s.shift(50).values
    gr_lag_100 = gr_norm_s.shift(100).values
    gr_lead_5  = gr_norm_s.shift(-5).values; gr_lead_10 = gr_norm_s.shift(-10).values
    gr_lead_25 = gr_norm_s.shift(-25).values

    eval_original_idx = hw[~anchor_mask].index.values

    out = pd.DataFrame({
        'well_id':   well_id,
        'row_index': eval_original_idx,
        # Formation plane-fit KNN (primary)
        'tvt_form_full': tvt_form_full, 'tvt_form_late': tvt_form_late,
        'tvt_form_wls':  tvt_form_wls,  'tvt_form_egfdu': tvt_form_egfdu,
        'tvt_form_buda': tvt_form_buda, 'ancc_knn': ancc_eval,
        'b_full': b_full, 'b_late': b_late, 'b_wls': b_wls,
        'b_early': b_early, 'b_mid': b_mid,
        'knn_dist': knn_dist,
        # Per-formation TVT predictions (all 6 formations × 3 segments)
        **tvt_form_feats,
        # Per-formation b_well offsets
        **{f'bw_{fn}':       np.float32(form_b[fn]['full'])  for fn in FORMATION_COLS},
        **{f'bww_{fn}':      np.float32(form_b[fn]['wls'])   for fn in FORMATION_COLS},
        **{f'bw50_{fn}':     np.float32(form_b[fn]['late'])  for fn in FORMATION_COLS},
        **{f'bw_early_{fn}': np.float32(form_b[fn]['early']) for fn in FORMATION_COLS},
        **{f'bw_mid_{fn}':   np.float32(form_b[fn]['mid'])   for fn in FORMATION_COLS},
        # RowKNN features
        'knn_row_ANCC':           row_ancc_eval,
        'knn_row_ANCC_std':       row_ancc_std_eval,
        'knn_row_dist':           row_dist_eval,
        'knn_row_b_well':         np.float32(b_well_row),
        'knn_row_tvt_pred_delta': knn_row_tvt_pred_delta,
        # DenseANCCImputer features
        'dense_ancc':           dense_ancc_eval,
        'dense_ancc_std':       dense_std_eval,
        'dense_dist':           dense_dist_eval,
        'dense_b_well':         np.float32(b_well_dense),
        'dense_tvt_pred_delta': dense_tvt_pred_delta,
        # GR signal
        'GR_imputed': gr_eval, 'GR_rolling_mean_25': roll_s_mean[n_anchor:],
        'GR_rolling_std_25': roll_s_std[n_anchor:], 'GR_rolling_mean_100': roll_l_mean[n_anchor:],
        'GR_rolling_std_100': roll_l_std[n_anchor:],
        'GR_typewell_at_anchor': gr_tw_last_anchor, 'GR_diff': gr_diff,
        'GR_norm_p5p95': gr_norm_p5p95,
        # GR derivatives and envelope
        'gr_d1': gr_d1, 'gr_d2': gr_d2, 'gr_env': gr_env,
        # xcorr
        'GR_xcorr_tvt': xcorr_tvt, 'GR_xcorr_conf': xcorr_conf,
        'GR_xcorr_delta': xcorr_delta, 'GR_xcorr_tvt_smooth': xcorr_smooth,
        'tw_gr_at_xcorr': tw_gr_at_xcorr, 'GR_xcorr_residual': xcorr_residual,
        # NCC vs typewell
        'GR_ncc_tvt': ncc_tvt, 'GR_ncc_conf': ncc_conf, 'GR_ncc_delta': ncc_delta,
        # NCC vs anchor zone (new)
        'GR_ncc_anchor_tvt':   ncc_anchor_tvt,
        'GR_ncc_anchor_conf':  ncc_anchor_conf,
        'GR_ncc_anchor_delta': ncc_anchor_delta,
        # Softmax blend
        'ncc_form_blend': ncc_form_blend, 'ncc_form_delta': ncc_form_delta,
        # Visible GR shift
        'visible_gr_shift_ft': gr_shift_ft, 'visible_gr_shift_corr': gr_shift_corr,
        'visible_gr_bias': gr_shift_bias,
        # Viterbi beam (4 variants)
        **beam_results,
        # Particle filter TVT (2 variants)
        **pf_results,
        # Particle filter ANCC (new)
        'pf_ancc_delta': pf_ancc_delta,
        'pf_ancc_std':   pf_ancc_std.astype(np.float32),
        # Signal divergence
        'sig_std':    sig_std,
        'form_std':   form_std,
        'pf_vs_spatial': pf_vs_spatial,
        'sc_vs_beam':    sc_vs_beam,
        # Typewell GR offset residuals anchored at NCC-anchor estimate
        **tdsc_feats,
        # Fractional position in eval zone
        'frac': frac.astype(np.float32),
        'frac2': frac2.astype(np.float32),
        'sqrt_frac': sqrt_frac.astype(np.float32),
        # GR lags/leads
        'GR_lag_5': gr_lag_5, 'GR_lag_10': gr_lag_10, 'GR_lag_25': gr_lag_25,
        'GR_lag_50': gr_lag_50, 'GR_lag_100': gr_lag_100,
        'GR_lead_5': gr_lead_5, 'GR_lead_10': gr_lead_10, 'GR_lead_25': gr_lead_25,
        # Typewell GR offsets
        **tw_diff,
        # Geometry
        'MD': md_ev, 'MD_from_anchor': md_from_anchor,
        'X': x_ev, 'Y': y_ev, 'Z': z_ev,
        'dZ_dMD': dz_dmd[n_anchor:], 'dX_dMD': dx_dmd[n_anchor:], 'dY_dMD': dy_dmd[n_anchor:],
        # Trajectory kinematics
        'incl_deg': incl_ev, 'azi_deg': azi_ev,
        'dls': dls_ev, 'build_rate': build_ev,
        'cos_incl': cos_incl_ev, 'sin_incl': sin_incl_ev,
        # Apparent dip from anchor zone
        'b_dip_full': np.float32(b_dip_full), 'b_dip_late': np.float32(b_dip_late),
        'b_dip_early': np.float32(b_dip_early), 'b_dip_slope': np.float32(b_dip_slope),
        'tvt_dip_full': tvt_dip_full_arr, 'tvt_dip_late': tvt_dip_late_arr,
        'azi_delta': azi_delta_ev, 'apparent_dip_dir': apparent_dip_dir,
        'plane_dip_x': np.float32(plane_dip_x), 'plane_dip_y': np.float32(plane_dip_y),

        # TVT extrapolation context
        'last_anchor_tvt': last_tvt, 'tvt_rate': tvt_rate, 'tvt_extrap': tvt_extrap,
        # Well context
        'anchor_gr_mean': anchor_gr_mean, 'anchor_gr_std': anchor_gr_std,
        'anchor_tvt_mean': anchor_tvt_mean, 'anchor_tvt_range': anchor_tvt_range,
        'n_anchor_rows': n_anchor_rows, 'tw_gr_anchor_mean': tw_gr_anchor_mean,
        # Prefix features
        'prefix_tvt_step20': prefix_tvt_step20, 'prefix_tvt_step100': prefix_tvt_step100,
        'prefix_tvt_md_slope100': prefix_tvt_md_slope100,
    })

    if 'TVT' in hw.columns:
        out['TVT'] = hw.loc[~anchor_mask, 'TVT'].values

    return out


In [14]:
with open(MODEL_DIR / 'feature_cols.json') as f:
    FEATURE_COLS = json.load(f)
print(f'Feature cols: {len(FEATURE_COLS)}')

test_ids = get_test_well_ids()
test_frames, errors = [], []

t0 = time.time()
for well_id in tqdm(test_ids, desc='Building test features'):
    try:
        feat = build_features(well_id, split='test', loo=False)
        if len(feat) > 0:
            test_frames.append(feat)
    except Exception as e:
        errors.append((well_id, str(e)))
        print(f'  ERROR {well_id}: {e}')

test_df = pd.concat(test_frames, ignore_index=True)
X_test  = test_df[FEATURE_COLS].values.astype(np.float32)
print(f'Test features: {test_df.shape}  |  {test_df["well_id"].nunique()} wells  ({time.time()-t0:.0f}s)')
print(f'NaN in X_test: {np.isnan(X_test).sum()}')

In [15]:
lgb_folds, xgb_folds, cb_folds = [], [], []

for fold_i in range(N_FOLDS):
    # LGB
    m = lgb.Booster(model_file=str(MODEL_DIR / f'lgb_fold{fold_i}.txt'))
    lgb_folds.append(m.predict(X_test).astype(np.float32))

    # XGB
    m = xgb.XGBRegressor()
    m.load_model(str(MODEL_DIR / f'xgb_fold{fold_i}.json'))
    xgb_folds.append(m.predict(X_test).astype(np.float32))

    # CB
    m = CatBoostRegressor()
    m.load_model(str(MODEL_DIR / f'cb_fold{fold_i}.cbm'))
    cb_folds.append(m.predict(X_test).astype(np.float32))

    print(f'  Fold {fold_i} loaded and predicted')

lgb_test_drift = np.mean(lgb_folds, axis=0)
xgb_test_drift = np.mean(xgb_folds, axis=0)
cb_test_drift  = np.mean(cb_folds,  axis=0)
print(f'LGB drift range: {lgb_test_drift.min():.2f}–{lgb_test_drift.max():.2f} ft')
print(f'XGB drift range: {xgb_test_drift.min():.2f}–{xgb_test_drift.max():.2f} ft')
print(f'CB  drift range: {cb_test_drift.min():.2f}–{cb_test_drift.max():.2f} ft')

In [ ]:
# ── Weighted blend ────────────────────────────────────────────────────────────
drift_preds = {'lgb': lgb_test_drift, 'xgb': xgb_test_drift, 'cb': cb_test_drift}
blend_drift = sum(BLEND_WEIGHTS[k] * drift_preds[k] for k in drift_preds).astype(np.float32)
print('Blend weights:', {k: f'{v:.4f}' for k, v in BLEND_WEIGHTS.items()})
print(f'Blended drift range: {blend_drift.min():.2f}–{blend_drift.max():.2f} ft')

# ── Savitzky-Golay smoothing per well (domain prior: w=17, poly=3) ────────────
def sg_smooth_wells(drift_arr, df, sg_w=SG_W, sg_p=SG_P):
    out = drift_arr.copy()
    for _, g in df.groupby('well_id', sort=False):
        idx   = g.sort_values('row_index').index
        chunk = drift_arr[idx]
        if len(chunk) >= sg_w:
            out[idx] = savgol_filter(chunk, sg_w, sg_p)
    return out

final_drift = sg_smooth_wells(blend_drift, test_df.reset_index(drop=True))
final_tvt   = (final_drift + test_df['last_anchor_tvt'].values.astype(np.float32))
print(f'Final TVT range: {final_tvt.min():.1f}–{final_tvt.max():.1f} ft')

In [ ]:
submission = test_df[['well_id', 'row_index']].copy()
submission['id'] = submission['well_id'].astype(str) + '_' + submission['row_index'].astype(str)
submission['tvt'] = final_tvt.astype(np.float32)
submission = submission[['id', 'tvt']]

template = pd.read_csv(utils.DATA_DIR / 'sample_submission.csv')[['id']]
template['id'] = template['id'].astype(str)
submission['id'] = submission['id'].astype(str)
if submission['id'].duplicated().any():
    raise RuntimeError('Duplicate submission ids')
missing = set(template['id']) - set(submission['id'])
extra = set(submission['id']) - set(template['id'])
if missing or extra:
    raise RuntimeError(f'Submission id mismatch: missing={len(missing)} extra={len(extra)}')
submission = template.merge(submission, on='id', how='left')
if submission['tvt'].isna().any():
    raise RuntimeError('Missing TVT values after template alignment')
if not np.isfinite(submission['tvt'].to_numpy(float)).all():
    raise RuntimeError('Non-finite TVT values')

out_path = Path('/kaggle/working/submission.csv') if _ON_KAGGLE else Path('submission.csv')
submission.to_csv(out_path, index=False)
print(f'Saved {out_path} ({len(submission)} rows)')
print(submission.head())
print(f'\nTVT stats: mean={submission["tvt"].mean():.1f} std={submission["tvt"].std():.1f} min={submission["tvt"].min():.1f} max={submission["tvt"].max():.1f}')
